In [ ]:
# 24i-6517
# Lab04

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats

np.random.seed(42)

n = 8000
regions = ['East', 'West', 'Central', 'South']
category_map = {
    'Furniture': ['Chairs', 'Tables', 'Bookcases', 'Furnishings'],
    'Office Supplies': ['Binders', 'Paper', 'Storage', 'Art'],
    'Technology': ['Phones', 'Machines', 'Accessories', 'Copiers']
}
categories = list(category_map.keys())

order_date = pd.to_datetime('2022-01-01') + pd.to_timedelta(np.random.randint(0, 730, n), unit='D')
region = np.random.choice(regions, n, p=[0.3, 0.25, 0.25, 0.2])
category = np.random.choice(categories, n, p=[0.25, 0.45, 0.3])
sub_category = [np.random.choice(category_map[c]) for c in category]
discount = np.round(np.random.choice([0, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5], n,
                                      p=[0.35, 0.15, 0.15, 0.15, 0.1, 0.06, 0.04]), 2)
sales = np.round(np.random.lognormal(mean=4.2, sigma=1.1, size=n), 2)
base_margin = np.where(category == 'Technology', 0.18,
              np.where(category == 'Furniture', 0.08, 0.22))
profit = np.round(sales * (base_margin - discount * 1.3) + np.random.normal(0, 15, n), 2)

df = pd.DataFrame({
    'order_date': order_date, 'region': region, 'category': category,
    'sub_category': sub_category, 'discount': discount, 'sales': sales, 'profit': profit
})
df = df.sort_values('order_date').reset_index(drop=True)
df.head()

In [ ]:
# Q1

vivid = LinearSegmentedColormap.from_list('vivid', ['#FF006E', '#FB5607', '#FFBE0B', '#8338EC', '#3A86FF'])

fig, ax = plt.subplots(figsize=(9, 5.5))
counts, bins, patches = ax.hist(df['sales'], bins=40, edgecolor='white', linewidth=0.6)
for c, p in zip(counts, patches):
    p.set_facecolor(vivid(c / counts.max()))

ax2 = ax.twinx()
sns.kdeplot(df['sales'], ax=ax2, color='#06D6A0', linewidth=3)
ax2.set_yticks([])

ax.set_xlabel('Sales')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Sales (Histogram + KDE)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

skew_val = df['sales'].skew()
mean_val = df['sales'].mean()
median_val = df['sales'].median()
print(f"Skewness: {skew_val:.3f}")
print(f"Mean: {mean_val:.2f}, Median: {median_val:.2f}")
if skew_val > 0.5:
    print("The sales distribution is right-skewed (mean > median), so the mean is pulled upward by a "
          "long tail of high-value orders and understates the typical transaction size.")
elif skew_val < -0.5:
    print("The sales distribution is left-skewed (mean < median), so the mean understates typical "
          "large values pulled down by a tail of low-value orders.")
else:
    print("The sales distribution is roughly symmetric, so the mean is a reasonable summary of a typical order.")

In [ ]:
# Q2

pal2 = sns.color_palette("husl", len(categories))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sns.violinplot(data=df, x='category', y='profit', hue='category', palette=pal2, legend=False, ax=axes[0])
axes[0].set_title('Profit by Category - Violin', fontweight='bold')
sns.stripplot(data=df, x='category', y='profit', hue='category', palette=pal2, legend=False,
              jitter=True, alpha=0.5, size=3, ax=axes[1])
axes[1].set_title('Profit by Category - Strip (jittered)', fontweight='bold')
plt.tight_layout()
plt.show()

spread = df.groupby('category')['profit'].std().sort_values(ascending=False)
top_cat = spread.index[0]
print(spread)
print(f"The violin plot better reveals spread/shape; '{top_cat}' shows the widest, most spread-out "
      f"profit distribution (std={spread.iloc[0]:.2f}), a pattern a bar-of-means would hide.")

In [ ]:
# Q3

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=df, x='discount', y='profit', hue='category', size='sales',
                 sizes=(15, 300), palette='rainbow', alpha=0.6, ax=ax)
ax.set_title('Discount vs Profit (hue=Category, size=Sales)', fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.show()

high_disc_big_sales = df[(df['discount'] >= 0.3) & (df['sales'] > df['sales'].quantile(0.75))]
print(f"Mean profit for large-sales, high-discount orders: {high_disc_big_sales['profit'].mean():.2f}")
print(f"Mean profit overall: {df['profit'].mean():.2f}")
print("The large-marker points (high sales) at high discount levels cluster at the lowest profit values, "
      "a pattern only visible once size encodes sales alongside color for category.")

In [ ]:
# Q4

num_cols = ['sales', 'discount', 'profit']
pearson_corr = df[num_cols].corr(method='pearson')
spearman_corr = df[num_cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Pearson Correlation', fontweight='bold')
sns.heatmap(spearman_corr, annot=True, cmap='PuOr', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Spearman Correlation', fontweight='bold')
plt.tight_layout()
plt.show()

diff = (pearson_corr - spearman_corr).abs()
diff_arr = np.array(diff.values, copy=True)
np.fill_diagonal(diff_arr, 0)
diff = pd.DataFrame(diff_arr, index=diff.index, columns=diff.columns)
pair = diff.stack().idxmax()
print(f"Largest disagreement: {pair} | Pearson={pearson_corr.loc[pair]:.3f}, Spearman={spearman_corr.loc[pair]:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x=pair[0], y=pair[1], alpha=0.4, color='#EF476F', ax=ax)
ax.set_title(f'{pair[0]} vs {pair[1]}', fontweight='bold')
plt.tight_layout()
plt.show()

print("The disagreement stems from a non-linear/noisy relationship and outlier profit values that "
      "distort the linear Pearson estimate while the rank-based Spearman stays robust to them.")
print("Correlation does not imply causation: an observed relationship here could reflect a third "
      "confounding variable or coincidence rather than a direct causal link.")

In [ ]:
# Q5

pivot = pd.pivot_table(df, index='region', columns='category', values='sales',
                        aggfunc='sum', margins=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='mako', linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Total Sales by Region x Category', fontweight='bold')
plt.tight_layout()
plt.show()

core = pivot.drop('All', axis=0).drop('All', axis=1)
best_region, best_cat = core.stack().idxmax()
print(f"Top contributor: {best_region} - {best_cat} (${core.loc[best_region, best_cat]:.0f})")

In [ ]:
# Q6

ts = df.set_index('order_date')['sales']
monthly = ts.resample('ME').sum()
rolling = monthly.rolling(window=3).mean()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(monthly.index, monthly.values, label='Monthly sales', color='#118AB2', linewidth=1.5, marker='o', markersize=4)
ax.plot(rolling.index, rolling.values, label='3-month rolling avg', color='#EF476F', linewidth=3)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='#118AB2')
ax.set_title('Monthly Sales: Trend vs Noise', fontweight='bold')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Monthly volatility (std): {monthly.std():.2f}")
print(f"Rolling average volatility (std): {rolling.std():.2f}")
print("The rolling average smooths out month-to-month spikes and dips, making the underlying upward or "
      "downward trend easier to see than in the noisier raw monthly line.")

In [ ]:
# Q7

df['profitable'] = np.where(df['profit'] > 0, 'Profitable', 'Loss')

g = sns.FacetGrid(df, col='region', hue='profitable', palette={'Profitable': '#06D6A0', 'Loss': '#EF476F'},
                   col_wrap=2, height=3.8, aspect=1.2)
g.map(sns.scatterplot, 'discount', 'profit', alpha=0.5, s=20)
g.add_legend()
g.fig.suptitle('Discount vs Profit by Region', fontweight='bold', y=1.03)
plt.show()

region_corr = df.groupby('region').apply(lambda d: d['discount'].corr(d['profit']))
print(region_corr)
diff_region = (region_corr - region_corr.mean()).abs().idxmax()
print(f"Region with the most different discount-profit relationship: {diff_region} (r={region_corr[diff_region]:.3f})")

In [ ]:
# Q8

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(df['discount'], df['profit'], alpha=1.0, s=10, color='#3A86FF')
axes[0].set_title('Raw Scatter (Overplotted)', fontweight='bold')
axes[0].set_xlabel('Discount')
axes[0].set_ylabel('Profit')

n_points = len(df)
use_hexbin = n_points >= 5000
if use_hexbin:
    hb = axes[1].hexbin(df['discount'], df['profit'], gridsize=30, cmap='plasma', mincnt=1)
    fig.colorbar(hb, ax=axes[1], label='Count')
    axes[1].set_title('Fixed: Hexbin', fontweight='bold')
else:
    axes[1].scatter(df['discount'], df['profit'], alpha=0.2, s=10, color='#3A86FF')
    axes[1].set_title('Fixed: Alpha Transparency', fontweight='bold')
axes[1].set_xlabel('Discount')
axes[1].set_ylabel('Profit')

best = df.loc[df['profit'].idxmax()]
worst = df.loc[df['profit'].idxmin()]
for a in axes:
    a.annotate(f"Max profit: {best['profit']:.0f}", xy=(best['discount'], best['profit']),
               xytext=(best['discount'] + 0.05, best['profit'] + 40),
               arrowprops=dict(arrowstyle='->', color='black'), fontsize=8)
    a.annotate(f"Largest loss: {worst['profit']:.0f}", xy=(worst['discount'], worst['profit']),
               xytext=(worst['discount'] + 0.05, worst['profit'] - 60),
               arrowprops=dict(arrowstyle='->', color='black'), fontsize=8)

plt.tight_layout()
plt.show()

print(f"Used {'hexbin' if use_hexbin else 'alpha transparency'} because the dataset has {n_points} points, "
      f"{'too many for transparency alone to resolve density' if use_hexbin else 'a moderate count where transparency suffices'}.")